In [ ]:
import os
import json
import ast
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, GradientBoostingClassifier, BaggingClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from lightgbm import LGBMClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score, roc_auc_score, cohen_kappa_score, matthews_corrcoef

In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 7.8 MB/s eta 0:00:00


In [ ]:
dfs = {
    'Printability' : 'Printability_resampled_df.csv',
    'Cell_Response' : 'Cell_Response_resampled_df.csv',
    'Scaffold_Quality' : 'Scaffold_Quality_(PxC)_resampled_df.csv'
}

In [ ]:
# Ensure models folder exists
os.makedirs('models', exist_ok=True)

for item in dfs.keys():
    df = pd.read_csv(dfs[item])

    X = df.iloc[:, :-1]
    y = df.iloc[:, -1:].values.ravel()

    unique_values = np.unique(y)
    value_to_continuous = {original_value: new_value for new_value, original_value in enumerate(unique_values)}
    y_continuous = np.array([value_to_continuous[value] for value in y])
    y = y_continuous

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

    # Scaling
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Convert back to DataFrame with original column names
    X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
    X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

    # Create a folder for each item under models
    model_folder = f'models/{item}'
    os.makedirs(model_folder, exist_ok=True)

    # Save the scaler
    joblib.dump(scaler, f'{model_folder}/scaler.pkl')


    # Setting up GridSearchCV parameters
    param_grid = {
        'BaggingClassifier': {
            'model': BaggingClassifier(random_state=42),
            'params': {'n_estimators': [10, 50, 100], 'max_samples': [0.5, 1.0], 'max_features': [0.5, 1.0]}
        },

        'ExtraTreesClassifier': {
            'model': ExtraTreesClassifier(random_state=42),
            'params': {
                'n_estimators': [50, 100],
                'max_depth': [None, 10],
                'criterion': ['gini', 'entropy'],
                'max_features': ['sqrt', 'log2'],
                'class_weight': ['balanced']}
        },

        'LightGBM': {
            'model': LGBMClassifier(random_state=42, verbose=-1),
            'params': {
                'n_estimators': [100],
                'boosting_type': ['gbdt', 'dart'],
                'class_weight': [None],
                'learning_rate': [0.01, 0.1],
                'max_depth': [3, 5],
                'num_leaves': [31]}
        },

        'GradientBoostingClassifier': {
            'model': GradientBoostingClassifier(random_state=42),
            'params': {
                'n_estimators': [100],
                'learning_rate': [0.01, 0.1],
                'max_depth': [3, 5],
                'loss': ['log_loss'],
                'criterion': ['friedman_mse'],
                'max_features': ['sqrt', 'log2']}
        },

        'RandomForestClassifier': {
            'model': RandomForestClassifier(n_jobs=-1, random_state=42),
            'params': {
                'n_estimators': [100, 200],
                'max_depth': [None, 10, 50],
                'criterion': ['gini', 'entropy'],
                'max_features': ['sqrt', 'log2'],
                'class_weight': ['balanced']}
        },

        'DecisionTreeClassifier': {
            'model': DecisionTreeClassifier(random_state=42),
            'params': {
                'max_depth': [None, 10],
                'min_samples_split': [2, 10],
                'criterion': ['gini', 'entropy']}

        },

        'CatBoost': {
            'model': CatBoostClassifier(verbose=0, random_state=42),
            'params': {
                'iterations': [100, 200],
                'learning_rate': [0.01, 0.1],
                'depth': [3, 5]
            }
        },

        'HistGradientBoostingClassifier': {
            'model': HistGradientBoostingClassifier(random_state=42),
            'params': {'max_iter': [100], 'learning_rate': [0.01, 0.1], 'max_depth': [None, 10],
                      'interaction_cst': ['pairwise']}
        },

        'SVC': {
            'model': SVC(random_state=42,probability=True),
            'params': {
                'C': [0.1, 1],
                'kernel': ['linear', 'rbf'],
                'gamma': ['scale']}

        },

        'KNeighborsClassifier': {
            'model': KNeighborsClassifier(),
            'params': {'n_neighbors': [3, 5, 7], 'weights': ['uniform', 'distance'],
                      'algorithm': ['auto', 'kd_tree']}
        },

         'LogisticRegression': {
            'model': LogisticRegression(solver='lbfgs', random_state=42),
            'params': {'C': [0.1, 1], 'penalty': ['l2'], 'class_weight': ['balanced']}
        },

         'XGBoost': {
            'model': XGBClassifier(random_state=42),
            'params': {
                'n_estimators': [100, 200], 'booster': ['gbtree'],
                'learning_rate': [0.01, 0.1], 'importance_type': ['gain', 'weight'],
                'max_depth': [3, 5]
            }
        },

    }


    # Cross-validation setup
    cv = StratifiedKFold(n_splits=10)

    # DataFrame for results
    result_df = pd.DataFrame(columns=['Classifier', 'Best Params', 'Precision', 'Recall',
                                      'Accuracy', 'F1 Score', 'AUC', 'Kappa', 'MCC'])


    for name, config in param_grid.items():
        grid_search = GridSearchCV(config['model'], config['params'], cv=cv, scoring='accuracy', verbose=0)
        grid_search.fit(X_train_scaled, y_train)

        # Predictions and evaluations
        y_pred = grid_search.predict(X_test_scaled)
        y_proba = grid_search.predict_proba(X_test_scaled)

        precision = precision_score(y_test, y_pred, average='macro')
        recall = recall_score(y_test, y_pred, average='macro')
        accuracy = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='macro')
        auc_score = roc_auc_score(y_test, y_proba, multi_class='ovr')
        kappa = cohen_kappa_score(y_test, y_pred)
        mcc = matthews_corrcoef(y_test, y_pred)

        # Update results DataFrame
        result_df.loc[len(result_df.index)] = [name, str(grid_search.best_params_), precision, recall, accuracy, f1, auc_score, kappa, mcc]


    # Save the results
    result_file = f'{model_folder}/results.csv'
    if os.path.exists(result_file):
        existing_df = pd.read_csv(result_file)
        # Drop duplicates based on the 'Classifier' and 'Best Params' columns
        combined_df = pd.concat([existing_df, result_df], ignore_index=True)
        combined_df = combined_df.drop_duplicates(subset=['Classifier', 'Best Params'], keep='last')
        combined_df.to_csv(result_file, index=False)
    else:
        result_df.to_csv(result_file, index=False)


In [ ]:
a=pd.read_csv(r'/content/models/Printability/results.csv')

In [ ]:
b=pd.read_csv(r'/content/models/Cell_Response/results.csv')

In [ ]:
c=pd.read_csv(r'/content/models/Scaffold_Quality/results.csv')

In [ ]:
print("Printability :\n")
a

Printability :



,Classifier,Best Params,Precision,Recall,Accuracy,F1 Score,AUC,Kappa,MCC
0,BaggingClassifier,"{'max_features': 0.5, 'max_samples': 1.0, 'n_e...",0.942274,0.940912,0.940385,0.940737,0.993587,0.920445,0.921022
1,ExtraTreesClassifier,"{'class_weight': 'balanced', 'criterion': 'ent...",0.953914,0.952016,0.951923,0.952058,0.992701,0.935827,0.936451
2,LightGBM,"{'boosting_type': 'gbdt', 'class_weight': None...",0.942518,0.940780,0.940385,0.940621,0.991146,0.920449,0.921145
3,GradientBoostingClassifier,"{'criterion': 'friedman_mse', 'learning_rate':...",0.921233,0.920422,0.921154,0.920599,0.988462,0.894751,0.894902
4,RandomForestClassifier,"{'class_weight': 'balanced', 'criterion': 'ent...",0.940302,0.938127,0.938462,0.938133,0.995008,0.917860,0.918582
5,DecisionTreeClassifier,"{'criterion': 'entropy', 'max_depth': None, 'm...",0.926124,0.926305,0.926923,0.925893,0.950843,0.902488,0.902698
6,CatBoost,"{'depth': 5, 'iterations': 200, 'learning_rate...",0.891192,0.889690,0.890385,0.889944,0.979868,0.853680,0.854005
7,HistGradientBoostingClassifier,"{'interaction_cst': 'pairwise', 'learning_rate...",0.920035,0.919701,0.919231,0.919212,0.990851,0.892242,0.892669
8,SVC,"{'C': 1, 'gamma': 'scale', 'kernel': 'rbf'}",0.745416,0.731928,0.734615,0.735075,0.909591,0.644914,0.646563
9,KNeighborsClassifier,"{'algorithm': 'auto', 'n_neighbors': 3, 'weigh...",0.918866,0.916795,0.917308,0.917144,0.967130,0.889594,0.890030


In [ ]:
print("Cell Response :\n")
b

Cell Response :



,Classifier,Best Params,Precision,Recall,Accuracy,F1 Score,AUC,Kappa,MCC
0,BaggingClassifier,"{'max_features': 1.0, 'max_samples': 1.0, 'n_e...",0.829583,0.832476,0.831126,0.829555,0.963638,0.788747,0.789459
1,ExtraTreesClassifier,"{'class_weight': 'balanced', 'criterion': 'ent...",0.837631,0.840720,0.839404,0.838406,0.935789,0.799066,0.799393
2,LightGBM,"{'boosting_type': 'gbdt', 'class_weight': None...",0.831642,0.834589,0.832781,0.832087,0.963786,0.790800,0.791248
3,GradientBoostingClassifier,"{'criterion': 'friedman_mse', 'learning_rate':...",0.824742,0.828278,0.826159,0.825090,0.959850,0.782530,0.783210
4,RandomForestClassifier,"{'class_weight': 'balanced', 'criterion': 'gin...",0.834061,0.835520,0.834437,0.833728,0.960200,0.792853,0.793326
5,DecisionTreeClassifier,"{'criterion': 'entropy', 'max_depth': None, 'm...",0.801596,0.802890,0.804636,0.802079,0.878148,0.755409,0.755481
6,CatBoost,"{'depth': 5, 'iterations': 200, 'learning_rate...",0.772322,0.770545,0.766556,0.763649,0.950868,0.708427,0.711510
7,HistGradientBoostingClassifier,"{'interaction_cst': 'pairwise', 'learning_rate...",0.812452,0.818455,0.816225,0.813897,0.959018,0.770181,0.770849
8,SVC,"{'C': 1, 'gamma': 'scale', 'kernel': 'rbf'}",0.627086,0.598263,0.592715,0.596330,0.883559,0.491172,0.497600
9,KNeighborsClassifier,"{'algorithm': 'auto', 'n_neighbors': 3, 'weigh...",0.778155,0.782205,0.779801,0.778592,0.915249,0.724523,0.725362


In [ ]:
print("Scaffold Quality :\n")
c

Scaffold Quality :



,Classifier,Best Params,Precision,Recall,Accuracy,F1 Score,AUC,Kappa,MCC
0,BaggingClassifier,"{'max_features': 0.5, 'max_samples': 1.0, 'n_e...",0.868823,0.865276,0.866087,0.865294,0.983125,0.852660,0.852995
1,ExtraTreesClassifier,"{'class_weight': 'balanced', 'criterion': 'gin...",0.866139,0.864751,0.866087,0.864181,0.962496,0.852636,0.852878
2,LightGBM,"{'boosting_type': 'gbdt', 'class_weight': None...",0.868880,0.865353,0.866087,0.865175,0.983881,0.852671,0.853046
3,GradientBoostingClassifier,"{'criterion': 'friedman_mse', 'learning_rate':...",0.868690,0.866537,0.867826,0.865796,0.983682,0.854579,0.854937
4,RandomForestClassifier,"{'class_weight': 'balanced', 'criterion': 'gin...",0.869940,0.865234,0.866087,0.865504,0.984018,0.852655,0.853052
5,DecisionTreeClassifier,"{'criterion': 'entropy', 'max_depth': None, 'm...",0.840574,0.837278,0.838261,0.837174,0.917952,0.822029,0.822415
6,CatBoost,"{'depth': 5, 'iterations': 200, 'learning_rate...",0.802360,0.795304,0.794783,0.790444,0.977670,0.774216,0.775733
7,HistGradientBoostingClassifier,"{'interaction_cst': 'pairwise', 'learning_rate...",0.850019,0.845722,0.846957,0.845744,0.982294,0.831624,0.832026
8,SVC,"{'C': 1, 'gamma': 'scale', 'kernel': 'rbf'}",0.708993,0.629401,0.631304,0.598813,0.941410,0.594235,0.602879
9,KNeighborsClassifier,"{'algorithm': 'kd_tree', 'n_neighbors': 3, 'we...",0.798947,0.795878,0.798261,0.794715,0.939207,0.778021,0.778648
